In [ ]:
import json
import pandas as pd

CORDOBA_ID = 2850

with open("../data/raw/cordoba_25_26_con_stats.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
matches = data["matches"]

for match in matches:

    event = match["event"]

    if "statistics" not in match:
        continue

    stats_block = match["statistics"].get("statistics")
    if not stats_block:
        continue

    stats_items = stats_block[0]["groups"]

    is_home = event["homeTeam"]["id"] == CORDOBA_ID

    row = {
        "match_id": event["id"],
        "date": pd.to_datetime(event["startTimestamp"], unit="s"),
        "is_home": is_home,
        "goals_for": event["homeScore"]["current"] if is_home else event["awayScore"]["current"],
        "goals_against": event["awayScore"]["current"] if is_home else event["homeScore"]["current"],
    }

    # Extraer TODAS las estadísticas
    for group in stats_items:
        for item in group["statisticsItems"]:
            key = item["key"]

            if is_home:
                value = item.get("homeValue")
            else:
                value = item.get("awayValue")

            row[key] = value

    rows.append(row)

df = pd.DataFrame(rows).sort_values("date")

print(df.shape)
print(df.columns)
print(df.head())

In [ ]:
df["points"] = df.apply(
    lambda r: 3 if r["goals_for"] > r["goals_against"]
    else 1 if r["goals_for"] == r["goals_against"]
    else 0,
    axis=1
)

df["goal_diff"] = df["goals_for"] - df["goals_against"]
df["xg_diff"] = df["expectedGoals"] - df["goal_diff"]
df["shot_accuracy"] = df["shotsOnGoal"] / df["totalShotsOnGoal"]
df["big_chance_efficiency"] = df["bigChanceScored"] / df["bigChanceCreated"].replace(0, 1)
df["pass_accuracy"] = df["accuratePasses"] / df["passes"]

df

In [ ]:
summary = {}

summary["matches"] = len(df)
summary["points"] = df["points"].sum()
summary["ppg"] = df["points"].mean()
summary["goal_diff_total"] = df["goal_diff"].sum()
summary["goal_diff_per_game"] = df["goal_diff"].mean()
summary["avg_xg"] = df["expectedGoals"].mean()
summary["total_xg"] = df["expectedGoals"].sum()
summary["conversion"] = df["goals_for"].sum() / df["expectedGoals"].sum()
summary["avg_possession"] = df["ballPossession"].mean()
summary["big_chance_eff"] = df["big_chance_efficiency"].mean()
summary["pressure_index_avg"] = (
    df["totalTackle"] +
    df["interceptionWon"] +
    df["ballRecovery"]
).mean()

summary

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

CORDOBA_ID = 2850
TOTAL_MATCHES_SEASON = 42

# ===============================
# 1️⃣ CARGA DE DATOS
# ===============================

with open("../data/raw/cordoba_25_26_con_stats.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
matches = data["matches"]

for match in matches:

    event = match["event"]
    stats_block = match["statistics"].get("statistics")

    if not stats_block:
        continue

    stats_items = stats_block[0]["groups"]

    is_home = event["homeTeam"]["id"] == CORDOBA_ID

    row = {
        "match_id": event["id"],
        "date": pd.to_datetime(event["startTimestamp"], unit="s"),
        "is_home": is_home,
        "goals_for": event["homeScore"]["current"] if is_home else event["awayScore"]["current"],
        "goals_against": event["awayScore"]["current"] if is_home else event["homeScore"]["current"],
    }

    for group in stats_items:
        for item in group["statisticsItems"]:
            key = item["key"]
            value = item.get("homeValue") if is_home else item.get("awayValue")
            row[key] = value

    rows.append(row)

df = pd.DataFrame(rows).sort_values("date")

# ===============================
# 2️⃣ MÉTRICAS DERIVADAS
# ===============================

df["points"] = df.apply(
    lambda r: 3 if r["goals_for"] > r["goals_against"]
    else 1 if r["goals_for"] == r["goals_against"]
    else 0,
    axis=1
)

df["goal_diff"] = df["goals_for"] - df["goals_against"]
df["conversion"] = df["goals_for"] / df["expectedGoals"]
df["big_chance_efficiency"] = df["bigChanceScored"] / df["bigChanceCreated"].replace(0, 1)
df["pressure_index"] = df["totalTackle"] + df["interceptionWon"] + df["ballRecovery"]

# ===============================
# 3️⃣ IDENTIDAD TÁCTICA
# ===============================

avg_possession = df["ballPossession"].mean()
shots_pos_ratio = (df["totalShotsOnGoal"] / df["ballPossession"]).mean()

if avg_possession > 55:
    style = "Equipo dominante de posesión"
elif avg_possession < 45:
    style = "Equipo reactivo / transición"
else:
    style = "Equipo híbrido"

# ===============================
# 4️⃣ EFICIENCIA VS DOMINIO
# ===============================

df["possession_bin"] = pd.cut(df["ballPossession"], bins=[0,45,55,100])
performance_by_possession = df.groupby("possession_bin")["points"].mean()

# ===============================
# 5️⃣ SOBRE / INFRA RENDIMIENTO
# ===============================

total_xg = df["expectedGoals"].sum()
total_goals = df["goals_for"].sum()
conversion_ratio = total_goals / total_xg

if conversion_ratio > 1.15:
    efficiency_status = "Sobre-rendimiento ofensivo"
elif conversion_ratio < 0.9:
    efficiency_status = "Infra-rendimiento ofensivo"
else:
    efficiency_status = "Conversión sostenible"

# ===============================
# 6️⃣ ÍNDICE DE PRESIÓN
# ===============================

pressure_home = df[df["is_home"]]["pressure_index"].mean()
pressure_away = df[~df["is_home"]]["pressure_index"].mean()

# ===============================
# 7️⃣ MODELO PREDICTIVO
# ===============================

features = [
    "ballPossession",
    "expectedGoals",
    "bigChanceCreated",
    "totalShotsOnGoal",
    "pressure_index"
]

model_df = df.dropna(subset=features)

X = model_df[features]
y = model_df["points"].apply(lambda x: 1 if x == 3 else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier()
model.fit(X_train, y_train)

feature_importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

# ===============================
# 8️⃣ SIMULADOR MONTE CARLO
# ===============================

current_points = df["points"].sum()
matches_played = len(df)
matches_remaining = TOTAL_MATCHES_SEASON - matches_played
ppg = df["points"].mean()

simulations = 10000
simulated_totals = []

for _ in range(simulations):
    future_points = np.random.poisson(ppg * matches_remaining)
    simulated_totals.append(current_points + future_points)

simulated_totals = np.array(simulated_totals)

prob_ascenso_directo = np.mean(simulated_totals >= 80)
prob_playoff = np.mean(simulated_totals >= 70)

# ===============================
# 9️⃣ OUTPUT RESUMEN
# ===============================

print("=== IDENTIDAD ===")
print("Posesión media:", round(avg_possession,2))
print("Estilo:", style)

print("\n=== EFICIENCIA ===")
print("Conversión goles/xG:", round(conversion_ratio,2))
print(efficiency_status)

print("\n=== PRESIÓN ===")
print("Índice presión casa:", round(pressure_home,2))
print("Índice presión fuera:", round(pressure_away,2))

print("\n=== MODELO PREDICTIVO ===")
print(feature_importance)

print("\n=== SIMULACIÓN ===")
print("Probabilidad ascenso directo:", round(prob_ascenso_directo,2))
print("Probabilidad playoff:", round(prob_playoff,2))